# RAG Document Assistant — Day 4: Retrieval Quality & Source Citations
### Filtering weak matches, tuning chunking, and making answers cite their sources properly.

**Stack:** Google Gemini 2.5 Flash + Gemini Embeddings + LangChain + ChromaDB

**Recap of Day 3:** we built the full RAG chain — retrieve chunks, format them into context, fill a prompt template, generate an answer with Gemini. It works, but it has two weak spots: irrelevant chunks still get passed to the model as if they were good matches, and answers don't cite sources in a structured, reliable way.

**Today's goal:** fix both. By the end of today, low-quality retrievals get filtered out *before* they reach the model (saving cost and preventing weak-context hallucination), and every answer includes a clear "Sources:" section pointing back to the actual document/page it came from.

---

## Step 1 — Install Dependencies (same as Day 3)

In [1]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Setup & Reload Vector Store

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

PERSIST_DIR = "chroma_db"

embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)
vector_store = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings_model)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print(f"Reloaded vector store with {vector_store._collection.count()} chunks")

Reloaded vector store with 13 chunks


## Step 3 — Understanding Your Score Distribution First

Before picking an arbitrary threshold, look at real numbers. Run a few realistic queries and print the scores — this tells you what "good" vs "bad" actually looks like for *your* documents and embedding model, instead of guessing a threshold blindly.

Remember from Day 2: for Chroma's default distance metric, **lower score = more similar**.

In [3]:
def inspect_scores(query, k=5):
    results = vector_store.similarity_search_with_score(query, k=k)
    print(f'Query: "{query}"')
    for doc, score in results:
        print(f"  score={score:.4f}  source={doc.metadata.get('source')}  page={doc.metadata.get('page')}")
    print()

# Try one question that SHOULD be answerable from your documents...
inspect_scores("What is the main topic of this document?")

# ...and one that almost certainly is NOT covered
inspect_scores("What is the boiling point of mercury?")

Query: "What is the main topic of this document?"
  score=0.6245  source=data\Untitled document.pdf  page=1
  score=0.6315  source=data\Untitled document.pdf  page=0
  score=0.6620  source=data\Untitled document.pdf  page=1
  score=0.6836  source=data\Untitled document.pdf  page=2
  score=0.6998  source=data\Untitled document.pdf  page=0

Query: "What is the boiling point of mercury?"
  score=1.0046  source=data\Untitled document.pdf  page=2
  score=1.0050  source=data\Untitled document.pdf  page=2
  score=1.0183  source=data\Untitled document.pdf  page=1
  score=1.0247  source=data\Untitled document.pdf  page=2
  score=1.0314  source=data\Untitled document.pdf  page=0



Look at the two score sets above. You should see the relevant query cluster at noticeably lower (better) scores than the irrelevant one. Pick a threshold that sits *between* those two clusters — that number goes into Step 4. (A reasonable starting point is often around 0.7–0.8 for this embedding model, but **your printed numbers above are the real source of truth**, not this comment.)

## Step 4 — Retrieval With a Score Threshold

Now we filter: only chunks scoring *below* our threshold (i.e. genuinely close matches) get kept. If nothing passes, we return an empty list — and downstream, that becomes a clean "not found in the documents" response instead of the model trying to answer from irrelevant context.

In [4]:
SCORE_THRESHOLD = 0.8  # adjust based on what you saw in Step 3

def retrieve_filtered(query: str, k: int = 5, threshold: float = SCORE_THRESHOLD):
    results = vector_store.similarity_search_with_score(query, k=k)
    filtered = [(doc, score) for doc, score in results if score <= threshold]
    return filtered

# Test it
filtered_results = retrieve_filtered("What is the main topic of this document?")
print(f"{len(filtered_results)} chunk(s) passed the threshold")

filtered_results_irrelevant = retrieve_filtered("What is the boiling point of mercury?")
print(f"{len(filtered_results_irrelevant)} chunk(s) passed the threshold (expect 0 or very few)")

5 chunk(s) passed the threshold
0 chunk(s) passed the threshold (expect 0 or very few)


## Step 5 — Format Context WITH Citation Labels

Same idea as Day 3's `format_docs`, but now each chunk gets a clean, consistent citation tag like `[Source 1]` that we can ask the model to reference directly in its answer — and we keep a lookup so we can print the real filename/page afterward.

In [5]:
def format_docs_with_citations(filtered_results):
    if not filtered_results:
        return "NO_RELEVANT_CONTEXT", {}

    context_parts = []
    citation_map = {}
    for i, (doc, score) in enumerate(filtered_results, 1):
        tag = f"Source {i}"
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        citation_map[tag] = f"{source} (page {page})"
        context_parts.append(f"[{tag}]\n{doc.page_content}")

    return "\n\n".join(context_parts), citation_map

## Step 6 — Updated Prompt: Require Citations, Handle "No Context" Cleanly

Two upgrades from Day 3's prompt:
1. Explicit instruction to reference `[Source N]` tags in the answer
2. A hard rule for what to say when context is `NO_RELEVANT_CONTEXT` — so a filtered-out query gets a clean, honest response instead of an empty or awkward one

In [6]:
RAG_PROMPT_V2 = """You are a helpful assistant answering questions using ONLY the context provided below.

Rules:
- Base your answer strictly on the context. Do not use outside knowledge.
- Reference the relevant [Source N] tag(s) inline when you use information from them.
- If the context is exactly "NO_RELEVANT_CONTEXT", respond only with: "I don't have relevant information in the provided documents to answer that."
- Be concise and direct.

Context:
{context}

Question:
{question}

Answer:"""

prompt_v2 = ChatPromptTemplate.from_template(RAG_PROMPT_V2)

## Step 7 — Build the Improved Chain

Same LCEL pattern as Day 3, but now built around our filtered retrieval + citation-aware formatting. Because filtering and formatting both need custom Python logic (not just a plain retriever call), we wrap the whole retrieval step in a small function via `RunnableLambda`.

In [7]:
def retrieve_and_format(question: str):
    filtered = retrieve_filtered(question)
    context, citation_map = format_docs_with_citations(filtered)
    return {"context": context, "citation_map": citation_map}

def ask_v2(question: str):
    retrieval = retrieve_and_format(question)

    answer = (prompt_v2 | llm | StrOutputParser()).invoke({
        "context": retrieval["context"],
        "question": question
    })

    print(f'Q: {question}\n')
    print(f'A: {answer}\n')

    if retrieval["citation_map"]:
        print("Sources referenced:")
        for tag, label in retrieval["citation_map"].items():
            print(f"  {tag} -> {label}")

    return answer

## Step 8 — Test It: Relevant Question

In [8]:
ask_v2("What is the main topic of this document?")

Q: What is the main topic of this document?

A: The main topics discussed in the documents are Cloud Computing and Modern Software Development [Source 1], Artificial Intelligence and Its Impact on Modern Society [Source 2], and The Importance of Data Structures and Algorithms [Source 3].

Sources referenced:
  Source 1 -> data\Untitled document.pdf (page 1)
  Source 2 -> data\Untitled document.pdf (page 0)
  Source 3 -> data\Untitled document.pdf (page 1)
  Source 4 -> data\Untitled document.pdf (page 2)
  Source 5 -> data\Untitled document.pdf (page 0)


'The main topics discussed in the documents are Cloud Computing and Modern Software Development [Source 1], Artificial Intelligence and Its Impact on Modern Society [Source 2], and The Importance of Data Structures and Algorithms [Source 3].'

## Step 9 — Test It: Out-of-Scope Question (should cleanly decline, not guess)

In [9]:
ask_v2("What is the boiling point of mercury?")

Q: What is the boiling point of mercury?

A: I don't have relevant information in the provided documents to answer that.



"I don't have relevant information in the provided documents to answer that."

## Step 10 — Revisiting Chunk Size (Optional Tuning Pass)

If your retrieved chunks in earlier days felt too short (missing context) or too long (diluted with irrelevant text), this is the moment to adjust. Smaller chunks → more precise but can lose surrounding context. Larger chunks → more context but less precise matching. There's no universal right answer — it depends on your documents' structure (dense technical text vs. narrative prose).

If you want to experiment, rebuild the store with a new `chunk_size`/`chunk_overlap` (same code as Day 2's Step 5, just with different numbers and, ideally, a different `persist_directory` so you don't overwrite your working store while comparing).

In [10]:
# Example only — uncomment and adjust if you want to experiment with different chunking
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
#
# loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyPDFLoader)
# documents = loader.load()
#
# experimental_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
# experimental_chunks = experimental_splitter.split_documents(documents)
#
# experimental_store = Chroma.from_documents(
#     documents=experimental_chunks,
#     embedding=embeddings_model,
#     persist_directory="chroma_db_experimental"
# )
# print(f"Experimental store: {len(experimental_chunks)} chunks")

---
## Day 4 Wrap-Up

Today you closed the two biggest quality gaps from Day 3:

`Threshold-filtered retrieval → citation-labeled context → prompt that must cite sources or admit ignorance`

Your RAG assistant now behaves noticeably more trustworthy: irrelevant queries get a clean refusal instead of a hallucinated guess, and every grounded answer can point back to exactly which document/page it came from.

**What's still missing:** every `ask_v2()` call is still fully independent — the model has no memory of what you asked a moment ago, so natural follow-ups like "tell me more about that" don't work yet.

**Tomorrow (Day 5):** we add conversational memory, so you can have an actual back-and-forth chat with your documents instead of firing isolated one-off questions.

See `README_Day4.md` for the full write-up of today's concepts.